In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU name: Quadro T1000


In [2]:
import torch
from transformers import AutoTokenizer

print("Torch:", torch.__version__)

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
print("Tokenizer loaded!")

Torch: 2.7.1+cu118
Tokenizer loaded!


In [3]:
import pandas as pd
import numpy as np
import torch

from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
# dataset emotion (english)
emotion = load_dataset("dair-ai/emotion")

train_dataset = emotion["train"]
test_dataset = emotion["test"]

In [6]:
def remap_label(example):
    if example["label"] in [0, 3, 4]:  # sadness, anger, fear
        return {"label": 0}  # negative
    elif example["label"] == 1:  # joy
        return {"label": 2}  # positive
    else:
        return {"label": 1}  # neutral

train_dataset = train_dataset.map(remap_label)
test_dataset = test_dataset.map(remap_label)

In [7]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

In [8]:
def tokenize(example):
    return tokenizer(
        example["text"],   # pakai text asli dari emotion
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

In [9]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=3
)

model.to(device)  # 🔥 WAJIB untuk GPU

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768,

In [11]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }
    

In [ ]:
training_args = TrainingArguments(
    output_dir="../results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [14]:
trainer.train()

  0%|          | 0/3000 [00:00<?, ?it/s]

{'loss': 0.5967, 'grad_norm': 45.772037506103516, 'learning_rate': 1.6666666666666667e-05, 'epoch': 0.5}
{'loss': 0.3478, 'grad_norm': 3.3017473220825195, 'learning_rate': 1.3333333333333333e-05, 'epoch': 1.0}


  0%|          | 0/125 [00:00<?, ?it/s]

{'eval_loss': 0.20529870688915253, 'eval_accuracy': 0.9395, 'eval_f1': 0.93984749713676, 'eval_precision': 0.9403149578589903, 'eval_recall': 0.9395, 'eval_runtime': 58.0161, 'eval_samples_per_second': 34.473, 'eval_steps_per_second': 2.155, 'epoch': 1.0}
{'loss': 0.2026, 'grad_norm': 18.57709312438965, 'learning_rate': 1e-05, 'epoch': 1.5}
{'loss': 0.1638, 'grad_norm': 256.5583801269531, 'learning_rate': 6.666666666666667e-06, 'epoch': 2.0}


  0%|          | 0/125 [00:00<?, ?it/s]

{'eval_loss': 0.16150866448879242, 'eval_accuracy': 0.9475, 'eval_f1': 0.9457317871687394, 'eval_precision': 0.9457867080351, 'eval_recall': 0.9475, 'eval_runtime': 78.4331, 'eval_samples_per_second': 25.499, 'eval_steps_per_second': 1.594, 'epoch': 2.0}
{'loss': 0.1191, 'grad_norm': 0.35678404569625854, 'learning_rate': 3.3333333333333333e-06, 'epoch': 2.5}
{'loss': 0.1009, 'grad_norm': 5.9287872314453125, 'learning_rate': 0.0, 'epoch': 3.0}


  0%|          | 0/125 [00:00<?, ?it/s]

{'eval_loss': 0.13032524287700653, 'eval_accuracy': 0.9525, 'eval_f1': 0.9521303171775868, 'eval_precision': 0.9518699174793699, 'eval_recall': 0.9525, 'eval_runtime': 57.6472, 'eval_samples_per_second': 34.694, 'eval_steps_per_second': 2.168, 'epoch': 3.0}
{'train_runtime': 7817.0669, 'train_samples_per_second': 6.14, 'train_steps_per_second': 0.384, 'train_loss': 0.2551382090250651, 'epoch': 3.0}


TrainOutput(global_step=3000, training_loss=0.2551382090250651, metrics={'train_runtime': 7817.0669, 'train_samples_per_second': 6.14, 'train_steps_per_second': 0.384, 'total_flos': 3157361012736000.0, 'train_loss': 0.2551382090250651, 'epoch': 3.0})

In [15]:
trainer.evaluate()

  0%|          | 0/125 [00:00<?, ?it/s]

{'eval_loss': 0.13032524287700653,
 'eval_accuracy': 0.9525,
 'eval_f1': 0.9521303171775868,
 'eval_precision': 0.9518699174793699,
 'eval_recall': 0.9525,
 'eval_runtime': 62.3333,
 'eval_samples_per_second': 32.086,
 'eval_steps_per_second': 2.005,
 'epoch': 3.0}

In [ ]:
model.save_pretrained("../model")
tokenizer.save_pretrained("../model")

('./model\\tokenizer_config.json',
 './model\\special_tokens_map.json',
 './model\\tokenizer.json')

In [19]:
text = "Saya merasa sangat lelah dan tidak bersemangat"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

# 🔥 PINDAHKAN KE GPU
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model(**inputs)

probs = torch.nn.functional.softmax(outputs.logits, dim=1)

print("Probabilities:", probs)
print("Prediction:", torch.argmax(probs, dim=1))

Probabilities: tensor([[9.9964e-01, 1.5172e-04, 2.0561e-04]], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)
Prediction: tensor([0], device='cuda:0')
